In [0]:
%python
dbutils.widgets.text("process_datetime", "")
dbutils.widgets.text("runId", "")

process_datetime = dbutils.widgets.get("process_datetime")
runId = dbutils.widgets.get("runId")

In [0]:
%python
import json

In [0]:
CREATE OR REPLACE TEMPORARY VIEW news_transacciones AS 
    SELECT 
        fecha,
        tipoTran,
        id_cliente,
        descripcion_titulo,
        moneda,
        simbolo_titulo,
        cantidad,
        precio,
        id_transaccion,
        origen,
        fecha_auditoria
    FROM bronze.transacciones
    WHERE fecha_auditoria = :process_datetime

In [0]:
CREATE OR REPLACE TEMPORARY VIEW transacciones_normalizadas AS
SELECT 
    fecha,
    IF(dayofweek(fecha) IN (1,7), FALSE, TRUE) AS es_dia_habil,
    tipoTran AS tipo_transaccion,
    id_cliente,
    descripcion_titulo,
    moneda,
    REGEXP_REPLACE(TRIM(simbolo_titulo), '(D|C)$', '') AS simbolo_base, -- elimino los sufijos del final que representan la variante de liquidacion
    simbolo_titulo AS simbolo_original,
    CASE 
        WHEN simbolo_titulo LIKE '%D' THEN 'MEP'
        WHEN simbolo_titulo LIKE '%C' THEN 'CABLE'
        ELSE 'STANDARD'
    END AS variante_liquidacion,
    CASE 
        WHEN descripcion_titulo LIKE '%Cedear%' THEN 'CEDEAR'
        WHEN descripcion_titulo LIKE 'On %' THEN 'ON'
        WHEN descripcion_titulo LIKE '%BONO%' OR descripcion_titulo LIKE '%Bono Rep%' OR descripcion_titulo LIKE '%BONOS REP%' THEN 'BONO_SOBERANO'
        WHEN descripcion_titulo LIKE '%Letra%' OR descripcion_titulo LIKE 'Lt.%' OR descripcion_titulo LIKE 'L. Tes.%' THEN 'LETRA'
        WHEN descripcion_titulo LIKE '%BOPREAL%' OR descripcion_titulo LIKE '%Bono Prov%' OR descripcion_titulo LIKE '%Bono Pcia%' THEN 'BONO_PROVINCIAL'
        ELSE 'ACCION_LOCAL'
    END AS tipo_instrumento, -- de esta forma califico los intrumentos
    cantidad,
    precio,
    id_transaccion,
    origen,
    fecha_auditoria
FROM news_transacciones

In [0]:
%python
records_read = spark.sql("""
    SELECT count(1) FROM transacciones_normalizadas
""")
if records_read.first()[0] == 0:
    result = {
        "status": "SUCCESS",
        "records_read": 0,
        "records_written": 0,
        "error_message": None,
        "table_name": "silver.transacciones",
        "layer": "SILVER"
    }
    dbutils.notebook.exit(json.dumps(result))

In [0]:
%python
try:
    merge_transacciones = spark.sql("""
        MERGE INTO silver.transacciones t
        USING transacciones_normalizadas s
        ON t.id_transaccion = s.id_transaccion
        WHEN NOT MATCHED THEN 
        INSERT (
            fecha,
            es_dia_habil,
            tipo_transaccion,
            id_cliente,
            descripcion_titulo,
            moneda,
            simbolo_base,
            simbolo_original,
            variante_liquidacion,
            tipo_instrumento,
            cantidad,
            precio,
            id_transaccion,
            origen,
            fecha_auditoria
        ) VALUES (
            s.fecha,
            s.es_dia_habil,
            s.tipo_transaccion,
            s.id_cliente,
            s.descripcion_titulo,
            s.moneda,
            s.simbolo_base,
            s.simbolo_original,
            s.variante_liquidacion,
            s.tipo_instrumento,
            s.cantidad,
            s.precio,
            s.id_transaccion,
            s.origen,
            s.fecha_auditoria
        )
    """)
except Exception as e:
    result = {
        "status": "ERROR",
        "records_read": records_read.first()[0],
        "records_written": 0,
        "error_message": str(e),
        "table_name": "silver.transacciones"
    }
    dbutils.notebook.exit(json.dumps(result))

In [0]:
%python
result = {
    "status": "SUCCESS",
    "records_read": records_read.first()[0],
    "records_written": merge_transacciones.first()[0],
    "error_message": None,
    "table_name": "silver.transacciones",
    "layer": "SILVER"
}
dbutils.notebook.exit(json.dumps(result))